# Binary Text Classification with the IMDB Dataset

**Daily Challenge Submission**

This notebook builds a feedforward neural network to classify IMDB movie reviews as positive or negative. It covers preprocessing, model building, training, overfitting detection, retraining, and final evaluation.

---

## Step 1 — Preprocess the Data

The IMDB dataset ships with Keras and contains 50,000 reviews pre-encoded as integer sequences (each integer maps to a word index). We keep only the 10,000 most frequent tokens.

Because a neural network cannot consume variable-length integer lists, we convert each review into a fixed-size **binary vector** of length 10,000 (one-hot / multi-hot encoding). Index `i` is set to 1 if word `i` appears in the review, regardless of how many times.

We then carve out a **validation set** of 10,000 samples from the training split so we can monitor generalisation during training without touching the test set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ── Load ──────────────────────────────────────────────────────────────────────
NUM_WORDS = 10_000

(train_data, train_labels), (test_data, test_labels) = \
    keras.datasets.imdb.load_data(num_words=NUM_WORDS)

print(f"Training samples : {len(train_data)}")
print(f"Test samples     : {len(test_data)}")
print(f"Sample review (encoded): {train_data[0][:20]} ...")
print(f"Label           : {train_labels[0]}  (1 = positive, 0 = negative)")

In [ ]:
# ── Vectorise (multi-hot encoding) ────────────────────────────────────────────
def vectorize_sequences(sequences, dimension=10_000):
    """Convert a list of integer sequences into a binary matrix."""
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1
    return results

x_train_full = vectorize_sequences(train_data)
x_test       = vectorize_sequences(test_data)

y_train_full = train_labels.astype('float32')
y_test       = test_labels.astype('float32')

# ── Train / Validation split ──────────────────────────────────────────────────
VAL_SIZE = 10_000

x_val   = x_train_full[:VAL_SIZE]
x_train = x_train_full[VAL_SIZE:]
y_val   = y_train_full[:VAL_SIZE]
y_train = y_train_full[VAL_SIZE:]

print(f"x_train shape : {x_train.shape}")
print(f"x_val shape   : {x_val.shape}")
print(f"x_test shape  : {x_test.shape}")

**Why multi-hot encoding?**

Each review becomes a 10,000-dimensional sparse binary vector. This lets a standard `Dense` layer operate on fixed-size floating-point input. The trade-off is that word order and frequency are lost — we only know *whether* each word appeared. For a shallow network on a well-structured sentiment task this is sufficient, but recurrent or attention-based models would use richer representations.

**Why reserve validation from training data, not from the test set?**

The test set must remain unseen until final evaluation. Using it for validation would leak information and give an over-optimistic estimate of generalisation.

---
## Step 2 — Build the Model

**Architecture rationale:**

Our inputs are dense floating-point vectors and our labels are scalars in {0, 1}. The simplest effective architecture is a **stack of fully-connected Dense layers with ReLU activations**:

- **ReLU** introduces non-linearity cheaply and avoids the vanishing-gradient problem of sigmoid in hidden layers.
- **Two hidden layers of 16 units** give the network enough capacity to learn non-linear interactions among word co-occurrences without over-parameterising.
- **Sigmoid output** squashes the scalar logit to (0, 1), interpretable as P(positive review). A single unit is all we need for binary classification.
- **Binary cross-entropy** is the theoretically correct loss for a Bernoulli target with a sigmoid output — it directly minimises the negative log-likelihood.
- **RMSprop** is a good default for NLP: it adapts learning rates per-parameter and is robust to noisy gradients.

In [ ]:
from tensorflow.keras import layers, Sequential

def build_model():
    model = Sequential([
        layers.Input(shape=(NUM_WORDS,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1,  activation='sigmoid'),
    ], name="imdb_classifier")
    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

---
## Step 3 — Train the Model (20 Epochs)

We first train for 20 epochs to get the full learning curves, then use those curves to identify the optimal stopping point.

In [ ]:
EPOCHS     = 20
BATCH_SIZE = 512

history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

---
## Step 4 — Evaluate: Detect Overfitting via Learning Curves

In [ ]:
hist = history.history
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss
ax1.plot(epochs_range, hist['loss'],     'bo-', label='Training loss')
ax1.plot(epochs_range, hist['val_loss'], 'ro-', label='Validation loss')
ax1.set_title('Training vs Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Binary cross-entropy')
ax1.legend()

# Accuracy
ax2.plot(epochs_range, hist['accuracy'],     'bo-', label='Training accuracy')
ax2.plot(epochs_range, hist['val_accuracy'], 'ro-', label='Validation accuracy')
ax2.set_title('Training vs Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.suptitle('Learning curves — 20-epoch run', fontsize=13)
plt.tight_layout()
plt.show()

# Find the best validation epoch
best_epoch = int(np.argmin(hist['val_loss'])) + 1
print(f"Best validation loss at epoch: {best_epoch}")
print(f"Val loss  at best epoch : {hist['val_loss'][best_epoch-1]:.4f}")
print(f"Val acc   at best epoch : {hist['val_accuracy'][best_epoch-1]:.4f}")

**Overfitting analysis (TODO — fill in after running):**

> *(Replace this text)* Validation loss reaches its minimum at epoch ___ and then begins to rise, while training loss continues to fall. This divergence is the classic overfitting signature: the model is memorising training-set patterns that do not generalise. Validation accuracy peaks near epoch ___ and plateaus or degrades slightly thereafter. We will retrain with `OPTIMAL_EPOCHS = ___` to avoid this.

### Retrain with Optimal Number of Epochs

We rebuild the model from scratch (fresh random weights) and train only up to the epoch where validation loss was lowest.

In [ ]:
# Set this to the epoch number identified above
OPTIMAL_EPOCHS = best_epoch   # or override manually, e.g. OPTIMAL_EPOCHS = 4

# Retrain on full training data (train + val) for best generalisation before testing
model_final = build_model()

history_final = model_final.fit(
    x_train_full, y_train_full,   # use all 25k training samples
    epochs=OPTIMAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"\nRetrained for {OPTIMAL_EPOCHS} epochs on full training set.")

---
## Step 5 — Analyze Results: Final Test Evaluation

In [ ]:
test_loss, test_acc = model_final.evaluate(x_test, y_test, verbose=0)
print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")

In [ ]:
# Sample predictions to inspect confidence
sample_probs = model_final.predict(x_test[:10], verbose=0).flatten()
sample_preds = (sample_probs >= 0.5).astype(int)

print(f"{'True':>6}  {'Predicted':>10}  {'P(positive)':>12}")
print("-" * 35)
for true, pred, prob in zip(y_test[:10], sample_preds, sample_probs):
    sentiment = lambda x: 'positive' if x else 'negative'
    match = "✓" if true == pred else "✗"
    print(f"{sentiment(true):>8}  {sentiment(pred):>10}  {prob:>12.4f}  {match}")

In [ ]:
# ── Comparison: 20-epoch val metrics vs final test metrics ────────────────────
print("Summary")
print("=" * 45)
print(f"Val   loss  (epoch {best_epoch:2d})  : {hist['val_loss'][best_epoch-1]:.4f}")
print(f"Val   acc   (epoch {best_epoch:2d})  : {hist['val_accuracy'][best_epoch-1]*100:.2f}%")
print("-" * 45)
print(f"Test  loss  (retrained)  : {test_loss:.4f}")
print(f"Test  acc   (retrained)  : {test_acc*100:.2f}%")

**Results analysis (TODO — fill in after running):**

> *(Replace this text)* The retrained model achieves **__% test accuracy**, consistent with the __% validation accuracy observed at epoch ___. This alignment confirms that our validation set was a reliable proxy for held-out performance and that early stopping prevented the model from memorising training noise.
>
> **Training vs validation behaviour:** During the 20-epoch run, training loss decreased monotonically while validation loss reached a minimum at epoch ___ before rising — a clear overfitting pattern. Retraining for exactly ___ epochs eliminates the memorisation phase.
>
> **Model limitations:** Multi-hot encoding discards word order and frequency information. A recurrent (LSTM) or attention-based (Transformer) model could capture phrase-level sentiment cues (e.g. "not good") that this architecture misses.
>
> **Threshold:** The default 0.5 threshold is reasonable for a balanced dataset. If false negatives (missed negative reviews) were more costly, lowering the threshold would increase recall for the negative class.

---
*End of notebook*